In [50]:
import fitz
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma,FAISS
from dotenv import load_dotenv

load_dotenv()

True

In [51]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [52]:
def embed_image(image_date):
    if isinstance(image_date, str):
        image = Image.open(image_date).convert("RGB")
    else:
        image = image_date
    
    inputs = clip_processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        
        # Extract the tensor from the output object
        features = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs
        
        # Now normalize the tensor
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


def embed_text(text):
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77
    )
    with torch.no_grad():
        # Get the output object
        outputs = clip_model.get_text_features(**inputs)
        
        # Extract the tensor (pooler_output) if it's an object, otherwise use as-is
        features = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs
        
        # Now normalize the actual tensor
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


In [53]:
pdf_path = r"C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\pdf_files\attention.pdf"
doc = fitz.open(pdf_path)
all_docs = []
all_embeddings = []
image_data_store = {}

splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [54]:
doc

Document('C:\Users\mendh\Langchain-Langgraph\0-Dataingestion-parsing\data\pdf_files\attention.pdf')

In [55]:
for i,page in enumerate(doc):
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text,metadata={"page":i,"type":"text"})
        text_chunks = splitter.split_documents([temp_doc])

        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk) 



    
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            image_id = f"page-{i}_img_{img_index}"

            buffered = io.BytesIO()

            pil_image.save(buffered,format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            image_data_store[image_id] = img_base64

            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            image_doc = Document(
                page_content=f"[Image:{image_id}]",
                metadata = {"page":i,"type":"image","image_id":image_id}
            )

            all_docs.append(image_doc)


        except Exception as e:
            print(f'error processing image {img_index} on page {i}: {e}')
            continue

doc.close()




In [56]:
embedding_array = np.array(all_embeddings)
embedding_array

array([[ 0.03500307,  0.0081093 , -0.03526161, ...,  0.02110538,
         0.0016752 , -0.03087725],
       [ 0.00678682, -0.00609661, -0.04333368, ..., -0.03352804,
        -0.02618883, -0.01386965],
       [ 0.00037512, -0.02122332, -0.00457744, ..., -0.01093405,
         0.03803127, -0.02663145],
       ...,
       [-0.01654971, -0.04369019,  0.0383744 , ..., -0.01301256,
         0.00746527, -0.07073216],
       [-0.01928524,  0.00096111,  0.00146912, ..., -0.03778004,
         0.02002626,  0.00973883],
       [-0.01122872, -0.02954055,  0.02860657, ..., -0.03015949,
         0.0220902 ,  0.00036676]], shape=(106, 512), dtype=float32)

In [57]:
(all_docs,all_embeddings)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu'),
  Document(metadata={'page': 0, 'type': 'text'}, page_content='Google Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. T

In [58]:
vector = FAISS.from_embeddings(
    text_embeddings = [(doc.page_content,emb) for doc, emb in zip(all_docs,embedding_array)],
    embedding=None,
    metadatas = [doc.metadata for doc in all_docs]
)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [59]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [60]:
llm = init_chat_model(model_provider='Groq',model="meta-llama/llama-4-scout-17b-16e-instruct")
llm.invoke("HI")

AIMessage(content='Hello! How are you today? Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 11, 'total_tokens': 34, 'completion_time': 0.054923392, 'completion_tokens_details': None, 'prompt_time': 0.000208147, 'prompt_tokens_details': None, 'queue_time': 0.063821372, 'total_time': 0.055131539}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4d7d-5cab-7590-956f-4c1ede888f46-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34})

In [61]:
def retriver_multimodel(query,k=5):
    query_embedding = embed_text(text)
    results = vector.similarity_search_by_vector(embedding=query_embedding,k=k)

    return results

In [64]:
def create_multimodel_message(query,retrived_docs):
    content = []
    content.append({
        "type":"text",
        "text":f'question: {query}\n\ncontext:\n'
    })

    text_docs = [doc for doc in retrived_docs if doc.metadata.get("type")=="text"]
    image_docs = [doc for doc in retrived_docs if doc.metadata.get("type")=="image"]

    if text_docs:
        text_context = "\n\n".join([
            f"[page {doc.metadata['page']}]:{doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type":'text',
            'text':f"text excerpts:\n{text_context}\n"
        })

    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type":"text",
                "text":f"\n[Image from page{doc.metadata["page"]}]"
            })
            content.append({
                "type":"image_url",
                "image_url":{"url":f"data:image/png;base64,{image_data_store[image_id]}"}
            })

    content.append(
        {
            "type":"text",
            "text":"\n\n Please answer the question based on the provided text and images."
        }
    )

    return HumanMessage(content=content)


In [65]:
context = retriver_multimodel("What is attention")
create_multimodel_message("What is Attention",context)

HumanMessage(content=[{'type': 'text', 'text': 'question: What is Attention\n\ncontext:\n'}, {'type': 'text', 'text': 'text excerpts:\n[page 13]:The\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n\n[page 14]:The\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis

In [66]:
def multimodal_pdf_rag_pipeline(query):
    context_docs = retriver_multimodel(query,k=5)

    message = create_multimodel_message(query,context_docs)

    response = llm.invoke([message])

    print(f"\nRetrived {len(context_docs)} documents:")

    for doc in context_docs:
        doc_type = doc.metadata.get("type","unknown")
        page = doc.metadata.get("page","?")
        if doc_type =="text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f" - Text from page {page}: {preview}")
        else:
            print(f" - image from page {page}")

    print("\n")

    return response.content



In [69]:
queries = [
    "What is Model Architecture?",
    "What is single head attention?",
    "What is multi head attention?",
    "What is Encoder-Decoder stack"
]

for query in queries:
    print(f"\n{query}")
    print("--"*50)
    answer = multimodal_pdf_rag_pipeline(
        query=query
    )
    print(f"response: {answer}")

    print("=="*50)


What is Model Architecture?
----------------------------------------------------------------------------------------------------

Retrived 5 documents:
 - Text from page 13: The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
i...
 - Text from page 14: The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
i...
 - Text from page 14: be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS...
 - Text from page 8: (section 5.4), learning rates and beam size on the Section 22 development set, all other parameters
...
 - Text from page 12: Attention Visualizations
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
ne...


response: Based on the provided text excerpts, I couldn't find a direct mention of "Model Architecture". However, I can infer some information related to model architecture from the text.

It appears that the text is